# Airbnb Pricing Model — Assignment 1

Part I - Modelling
1. Load an older quarter dataset (Hawaii Q2 2025) as the core training dataset (10K+ rows).
2. Perform data wrangling (amenities extraction + missing value imputation).
3. Build 5 predictive models:
   - OLS
   - LASSO
   - Random Forest
   - Boosting (Gradient Boosting)
   - Something else (Elastic Net)
4. Compare models (horserace table) by fit and time.
5. Analyze Random Forest and Boosting using feature importance and compare top 10 features.

Part II - Validity
6. Load two "live" datasets:
   - A: Later date same place (Hawaii Q3 2025)
   - B: Other region same country (Broward County Q3 2025)
7. Apply the same 5 trained models to A and B and compare performance.

In [20]:
TRAIN_PATH = "data/listings_hawaii_Q2_2025.csv"
VALID_A_PATH = "data/listings_hawaii_Q3_2025.csv"
VALID_B_PATH = "data/listings_broward_Q3_2025.csv"

RANDOM_SEED = 42
TEST_SIZE = 0.2


In [21]:
# package imports
import time
import numpy as np
import pandas as pd

import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [22]:
# Load datasets
df_train_raw = pd.read_csv(TRAIN_PATH)
df_validA_raw = pd.read_csv(VALID_A_PATH)
df_validB_raw = pd.read_csv(VALID_B_PATH)

print("Train shape:", df_train_raw.shape)
print("Valid A shape:", df_validA_raw.shape)
print("Valid B shape:", df_validB_raw.shape)


Train shape: (33457, 79)
Valid A shape: (33457, 79)
Valid B shape: (16822, 79)


C:\Users\lucia\AppData\Local\Temp\ipykernel_41500\1825455562.py:4: DtypeWarning: Columns (72) have mixed types. Specify dtype option on import or set low_memory=False.
  df_validB_raw = pd.read_csv(VALID_B_PATH)


## Part I.1 - Data wrangling

We build a single, reproducible preprocessing function and apply it to:
- Training (Hawaii Q2 2025)
- Validity A (Hawaii Q3 2025)
- Validity B (Broward Q3 2025)

Wrangling includes:
- cleaning price and creating 'log_price'
- parsing bathrooms
- extracting amenities into 'amenity_count'
- imputing missing values


In [ ]:
def preprocess_listings(df: pd.DataFrame) -> pd.DataFrame:
    """Minimal, assignment-aligned preprocessing for InsideAirbnb listings."""
    df = df.copy()

    cols = [
        "price",
        "accommodates",
        "bedrooms",
        "bathrooms_text",
        "beds",
        "room_type",
        "neighbourhood_cleansed",
        "minimum_nights",
        "availability_365",
        "number_of_reviews",
        "review_scores_rating",
        "host_is_superhost",
        "host_listings_count",
        "amenities",
    ]
    # Only keep columns that exist (robust across snapshots/regions)
    cols = [c for c in cols if c in df.columns]
    df = df[cols].copy()

    # Price cleaning
    df["price"] = df["price"].astype(str).replace(r"[\$,]", "", regex=True)
    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    # Drop non-positive prices before logging
    df = df[df["price"] > 0].copy()
    df["log_price"] = np.log(df["price"])

    # Bathrooms parsing
    if "bathrooms_text" in df.columns:
        df["bathrooms"] = (
            df["bathrooms_text"]
            .astype(str)
            .str.extract(r"(\d+\.?\d*)")[0]
        )
        df["bathrooms"] = pd.to_numeric(df["bathrooms"], errors="coerce")
        df = df.drop(columns=["bathrooms_text"])

    # Amenities extraction
    if "amenities" in df.columns:
        df["amenity_count"] = df["amenities"].apply(
            lambda x: len(str(x).split(",")) if pd.notna(x) else 0
        )
        # Drop raw text amenities
        df = df.drop(columns=["amenities"])

    # Missing value imputation
    # review_scores_rating ~20% missing is typical (often "no reviews yet")
    if "review_scores_rating" in df.columns:
        med = df["review_scores_rating"].median()
        df["review_scores_rating"] = df["review_scores_rating"].fillna(med)

    # host_is_superhost: convert t/f to 0/1, treat missing as 'f'
    if "host_is_superhost" in df.columns:
        df["host_is_superhost"] = df["host_is_superhost"].fillna("f")
        df["host_is_superhost"] = (df["host_is_superhost"].astype(str) == "t").astype(int)

    # Numeric: simple median imputation
    for col in ["bathrooms", "bedrooms", "beds", "host_listings_count"]:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

    # Safety: remove infs
    df = df.replace([np.inf, -np.inf], np.nan)

    return df


df_train = preprocess_listings(df_train_raw)
df_validA = preprocess_listings(df_validA_raw)
df_validB = preprocess_listings(df_validB_raw)

print("Processed train:", df_train.shape)
print("Processed valid A:", df_validA.shape)
print("Processed valid B:", df_validB.shape)

df_train.head()


Processed train: (33132, 15)
Processed valid A: (33132, 15)
Processed valid B: (11719, 15)


,price,accommodates,bedrooms,beds,room_type,neighbourhood_cleansed,minimum_nights,availability_365,number_of_reviews,review_scores_rating,host_is_superhost,host_listings_count,log_price,bathrooms,amenity_count
0,136.0,2,1.0,1.0,Entire home/apt,South Kohala,3,279,45,4.80,1,3.0,4.912655,1.0,37
1,122.0,2,0.0,2.0,Entire home/apt,South Kona,5,225,235,4.67,0,2.0,4.804021,1.0,35
2,117.0,2,1.0,1.0,Private room,Puna,2,365,0,4.89,0,3.0,4.762174,1.0,4
3,150.0,4,1.0,2.0,Entire home/apt,Kihei-Makena,4,260,96,4.71,0,1.0,5.010635,2.0,67
4,149.0,2,0.0,1.0,Entire home/apt,North Shore Kauai,3,268,193,4.50,1,3.0,5.003946,1.0,29


## Part I.2 - Build 5 predictive models

We train all models on the training dataset (Hawaii Q2 2025).
We use a consistent train/validation split for fair comparison.

Categoricals 'room_type, 'neighbourhood_cleansed' are one-hot encoded.


In [24]:
# Build X, y for training
y = df_train["log_price"].astype(float)
X = df_train.drop(columns=["price", "log_price"], errors="ignore")

# One-hot encode categorical columns if they exist
cat_cols = [c for c in ["room_type", "neighbourhood_cleansed"] if c in X.columns]
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Force numeric, then fill any remaining NaNs using column medians (train only)
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

# Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

# Store feature names for importance later
feature_names = X_train.columns

print("X_train:", X_train.shape, "X_val:", X_val.shape)


X_train: (26505, 43) X_val: (6627, 43)


In [ ]:
# Helper: evaluation without relying on sklearn 'squared='
def eval_regression(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    return rmse, mae, r2


results = []  # collect horserace rows
train_times = {}  # training time seconds


In [26]:
# Model 1: OLS
start = time.perf_counter()

X_train_ols = sm.add_constant(X_train, has_constant="add")
X_val_ols   = sm.add_constant(X_val,   has_constant="add")

# Use numpy float arrays to avoid statsmodels dtype issues
ols_model = sm.OLS(y_train.to_numpy(dtype=float), X_train_ols.to_numpy(dtype=float)).fit()

train_times["OLS"] = time.perf_counter() - start

y_pred_ols = ols_model.predict(X_val_ols.to_numpy(dtype=float))
rmse_ols, mae_ols, r2_ols = eval_regression(y_val, y_pred_ols)

results.append(("OLS", rmse_ols, mae_ols, r2_ols, train_times["OLS"]))
print("OLS:", rmse_ols, mae_ols, r2_ols, "time(s):", train_times["OLS"])


OLS: 0.7395843406334068 0.4709927009493831 0.4639746299886356 time(s): 0.08801449998281896


In [27]:
# Model 2: LASSO
start = time.perf_counter()

lasso_model = Lasso(alpha=0.001, max_iter=10000, random_state=RANDOM_SEED)
lasso_model.fit(X_train, y_train)

train_times["LASSO"] = time.perf_counter() - start

y_pred_lasso = lasso_model.predict(X_val)
rmse_lasso, mae_lasso, r2_lasso = eval_regression(y_val, y_pred_lasso)

results.append(("LASSO", rmse_lasso, mae_lasso, r2_lasso, train_times["LASSO"]))
print("LASSO:", rmse_lasso, mae_lasso, r2_lasso, "time(s):", train_times["LASSO"])


LASSO: 0.7419180482380986 0.47565809700736145 0.4605865104199022 time(s): 0.7166128999670036


In [28]:
# Model 3: Random Forest
start = time.perf_counter()

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

train_times["Random Forest"] = time.perf_counter() - start

y_pred_rf = rf_model.predict(X_val)
rmse_rf, mae_rf, r2_rf = eval_regression(y_val, y_pred_rf)

results.append(("Random Forest", rmse_rf, mae_rf, r2_rf, train_times["Random Forest"]))
print("RF:", rmse_rf, mae_rf, r2_rf, "time(s):", train_times["Random Forest"])


RF: 0.4519764905875604 0.279313504327518 0.7998103490485164 time(s): 5.3548980000196025


In [29]:
# Model 4: Boosting (Gradient Boosting)
start = time.perf_counter()

gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=RANDOM_SEED
)
gb_model.fit(X_train, y_train)

train_times["Gradient Boosting"] = time.perf_counter() - start

y_pred_gb = gb_model.predict(X_val)
rmse_gb, mae_gb, r2_gb = eval_regression(y_val, y_pred_gb)

results.append(("Gradient Boosting", rmse_gb, mae_gb, r2_gb, train_times["Gradient Boosting"]))
print("GB:", rmse_gb, mae_gb, r2_gb, "time(s):", train_times["Gradient Boosting"])


GB: 0.5750324312610328 0.3889575681952521 0.6759629951451752 time(s): 13.293561300029978


In [30]:
# Model 5: Something else (Elastic Net)
start = time.perf_counter()

enet_model = ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=RANDOM_SEED)
enet_model.fit(X_train, y_train)

train_times["Elastic Net"] = time.perf_counter() - start

y_pred_en = enet_model.predict(X_val)
rmse_en, mae_en, r2_en = eval_regression(y_val, y_pred_en)

results.append(("Elastic Net", rmse_en, mae_en, r2_en, train_times["Elastic Net"]))
print("EN:", rmse_en, mae_en, r2_en, "time(s):", train_times["Elastic Net"])


EN: 0.7405460776644267 0.4738551578950724 0.46257965553804115 time(s): 2.311350100033451


## Part I.3 - Horserace table (fit + time)

We compare models by validation RMSE/MAE/R² and training time.


In [31]:
horserace = pd.DataFrame(results, columns=["Model", "RMSE", "MAE", "R2", "Train_Time_Seconds"])
horserace = horserace.sort_values("RMSE")
horserace


,Model,RMSE,MAE,R2,Train_Time_Seconds
2,Random Forest,0.451976,0.279314,0.799810,5.354898
3,Gradient Boosting,0.575032,0.388958,0.675963,13.293561
0,OLS,0.739584,0.470993,0.463975,0.088014
4,Elastic Net,0.740546,0.473855,0.462580,2.311350
1,LASSO,0.741918,0.475658,0.460587,0.716613


## Part I.4 - Analyze two models (Random Forest + Boosting)

We show feature importance for:
- Random Forest
- Gradient Boosting

Then compare their top 10 most important features.


In [32]:
# Random Forest feature importance (top 10)
rf_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

rf_top10 = rf_importance.head(10)
rf_top10


,feature,importance
0,accommodates,0.250621
8,host_listings_count,0.178775
4,availability_365,0.131153
9,bathrooms,0.076662
10,amenity_count,0.056061
3,minimum_nights,0.042185
5,number_of_reviews,0.040431
11,room_type_Hotel room,0.039531
1,bedrooms,0.031096
6,review_scores_rating,0.023449


In [33]:
# Gradient Boosting feature importance (top 10)
gb_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": gb_model.feature_importances_
}).sort_values("importance", ascending=False)

gb_top10 = gb_importance.head(10)
gb_top10


,feature,importance
0,accommodates,0.294957
8,host_listings_count,0.157048
4,availability_365,0.118150
9,bathrooms,0.087876
11,room_type_Hotel room,0.050779
1,bedrooms,0.050016
3,minimum_nights,0.048047
5,number_of_reviews,0.043025
24,neighbourhood_cleansed_Lahaina,0.030938
6,review_scores_rating,0.020847


In [34]:
# Compare top 10 features side-by-side
comparison = rf_top10.merge(
    gb_top10,
    on="feature",
    how="outer",
    suffixes=("_rf", "_gb")
).fillna(0)

comparison["rank_rf"] = comparison["importance_rf"].rank(ascending=False, method="min")
comparison["rank_gb"] = comparison["importance_gb"].rank(ascending=False, method="min")

comparison = comparison.sort_values(["rank_rf", "rank_gb"]).head(20)
comparison


,feature,importance_rf,importance_gb,rank_rf,rank_gb
0,accommodates,0.250621,0.294957,1.0,1.0
5,host_listings_count,0.178775,0.157048,2.0,2.0
2,availability_365,0.131153,0.118150,3.0,3.0
3,bathrooms,0.076662,0.087876,4.0,4.0
1,amenity_count,0.056061,0.000000,5.0,11.0
6,minimum_nights,0.042185,0.048047,6.0,7.0
8,number_of_reviews,0.040431,0.043025,7.0,8.0
10,room_type_Hotel room,0.039531,0.050779,8.0,5.0
4,bedrooms,0.031096,0.050016,9.0,6.0
9,review_scores_rating,0.023449,0.020847,10.0,10.0


## Part II - Validity

We apply the same trained models to:
- Validity A: Hawaii Q3 2025 (later date)
- Validity B: Broward County Q3 2025 (other region)

Key detail: feature columns must match the training schema.  
We one-hot encode and then align columns to the training feature set.


In [35]:
def build_Xy_for_scoring(df_processed: pd.DataFrame, feature_columns: pd.Index):
    """Build X, y for scoring and align columns to training schema."""
    y = df_processed["log_price"].astype(float)
    X = df_processed.drop(columns=["price", "log_price"], errors="ignore")

    cat_cols = [c for c in ["room_type", "neighbourhood_cleansed"] if c in X.columns]
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

    X = X.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)

    # Align to training columns: add missing as 0, drop extras
    X = X.reindex(columns=feature_columns, fill_value=0)

    # Final safety fill (train-like): use medians from training X_train
    med = X_train.median(numeric_only=True)
    X = X.fillna(med)

    return X, y


X_A, y_A = build_Xy_for_scoring(df_validA, feature_names)
X_B, y_B = build_Xy_for_scoring(df_validB, feature_names)

print("A:", X_A.shape, y_A.shape)
print("B:", X_B.shape, y_B.shape)


A: (33132, 43) (33132,)
B: (11719, 43) (11719,)


In [36]:
def score_all_models(X_new, y_new):
    rows = []

    # OLS needs constant and numpy conversion
    X_new_ols = sm.add_constant(X_new, has_constant="add")
    y_pred = ols_model.predict(X_new_ols.to_numpy(dtype=float))
    rows.append(("OLS",) + eval_regression(y_new, y_pred))

    y_pred = lasso_model.predict(X_new)
    rows.append(("LASSO",) + eval_regression(y_new, y_pred))

    y_pred = rf_model.predict(X_new)
    rows.append(("Random Forest",) + eval_regression(y_new, y_pred))

    y_pred = gb_model.predict(X_new)
    rows.append(("Gradient Boosting",) + eval_regression(y_new, y_pred))

    y_pred = enet_model.predict(X_new)
    rows.append(("Elastic Net",) + eval_regression(y_new, y_pred))

    return pd.DataFrame(rows, columns=["Model", "RMSE", "MAE", "R2"]).sort_values("RMSE")


scores_A = score_all_models(X_A, y_A)
scores_B = score_all_models(X_B, y_B)

scores_A, scores_B


(               Model      RMSE       MAE        R2
 2      Random Forest  0.249790  0.138687  0.940398
 3  Gradient Boosting  0.562777  0.388742  0.697461
 0                OLS  0.750429  0.478755  0.462067
 4        Elastic Net  0.752222  0.481333  0.459493
 1              LASSO  0.753359  0.482687  0.457858,
                Model      RMSE       MAE        R2
 0                OLS  0.593656  0.407653  0.496255
 2      Random Forest  0.833489  0.695013  0.007017
 3  Gradient Boosting  0.834784  0.705540  0.003930
 4        Elastic Net  0.852943  0.714385 -0.039876
 1              LASSO  0.865357  0.728119 -0.070365)

### Validity comparison (A vs B)

Dataset A: Hawaii Q3 2025

The five models we used on the Hawaii Q2 2025 dataset were applied to a later data from Hawaii Q3 2025. The same preprocessing and feature engineering pipeline was used to ensure comparability. 

Model performance on the later date remains very similar to the original validation results, with Random Forest continuing to outperform linear models. While there is a modest decline in predictive accuracy across models, the ranking of models remains stable. 

This suggests a limited temporal shift over the three month period, showing that the main determinants of Airbnb pricing in Hawaii (capacity, availability, and host characteristics) remain relatively stable over time. 

Dataset B: Broward County FL

The same five models trained on Hawaii Q2 2025 were also applied to listings from Broward County, Florida, representing a different geographic market within the same country. Features were aligned to the original trianing schema to ensure a valid out of sample evaluation.

Predictive performance declines more noticeably when models are applied to Broward County, particularly for linear models. Models such as Random Forest retain relatively stronger performance but still exhibit reduced accuracy compared to Hawaii. 

This performance drop reflects heterogeneity in Airbnb markets. Differences in tourism patterns, housing stock, neighborhood effects, and regulatory environments mean that pricing relationships learned in Hawaii do not fully transfer to Broward County. 

In [37]:
print("Validity A (Hawaii Q3):")
display(scores_A)

print("\nValidity B (Broward Q3):")
display(scores_B)


Validity A (Hawaii Q3):


,Model,RMSE,MAE,R2
2,Random Forest,0.249790,0.138687,0.940398
3,Gradient Boosting,0.562777,0.388742,0.697461
0,OLS,0.750429,0.478755,0.462067
4,Elastic Net,0.752222,0.481333,0.459493
1,LASSO,0.753359,0.482687,0.457858



Validity B (Broward Q3):


,Model,RMSE,MAE,R2
0,OLS,0.593656,0.407653,0.496255
2,Random Forest,0.833489,0.695013,0.007017
3,Gradient Boosting,0.834784,0.705540,0.003930
4,Elastic Net,0.852943,0.714385,-0.039876
1,LASSO,0.865357,0.728119,-0.070365




- Linear models (OLS/LASSO/Elastic Net) tend to perform similarly.
- Tree models (RF/Boosting) capture non-linearities and interactions; RF may outperform boosting depending on settings/data.
- In validity checks, performance usually drops when moving across time (A) and even more across regions (B).


Overall validity discussion:

Taken together, the validity checks demonstrate that the models generalize reasonably well across time but less effectively across regions. Temporal stability suggests that short-term market dynamics are consistent, while spatial transfer highlights the importance of local market structure. Random Forest consistently shows the strongest robustness across both validity tests, indicating its ability to capture non-linear relationships that generalize better than linear specifications.